# Notebook 03 — Generative LoRA Verdict Generation

**Goal**: Fine-tune `meta-llama/Llama-3.1-8B-Instruct` with LoRA (PEFT) for veracity verdict generation.

**Config**: 4-bit quantisation (bitsandbytes), LoRA r=16, α=16, dropout=0.05, targeting attention modules.

**Baselines**: Zero-Shot and 5-step Chain-of-Thought prompting on the untuned base model.

**Output schema**: model verdict label + Romanian justification text.

In [ ]:
# %pip install transformers peft bitsandbytes accelerate datasets trl pandas

In [ ]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
import torch

sys.path.insert(0, str(Path('..').resolve()))
from src.metrics import classification_metrics, print_classification_metrics
from src.tools import parse_verdict

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../data/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

BASE_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
LORA_OUTPUT = str(MODELS_DIR / 'llama_lora_verdict')

## 1. Load Data

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')

LABEL_RO = {'true': 'ADEVĂRAT', 'partially_true': 'PARȚIAL_ADEVĂRAT', 'false': 'FALS'}

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 2. Prompt Templates

In [ ]:
SYSTEM_PROMPT = (
    'Ești un expert verificator de fapte pentru Republica Moldova. '
    'Evaluezi afirmații în română pe baza dovezilor furnizate. '
    'Răspunzi EXCLUSIV în baza contextului dat, fără cunoștințe externe.'
)

VERDICT_PROMPT_TEMPLATE = """\
Afirmație: {claim}

Dovezi:
{evidence}

Analizează afirmația pe baza dovezilor și oferă:
Verdict: [ADEVĂRAT / FALS / PARȚIAL_ADEVĂRAT]
Justificare: [explicație concisă în română, max 3 propoziții]"""


def build_training_prompt(row: pd.Series) -> str:
    label_ro = LABEL_RO.get(str(row.get('veracity_label', '')), 'NECUNOSCUT')
    user_part = VERDICT_PROMPT_TEMPLATE.format(
        claim=row.get('claim_text', ''),
        evidence=row.get('evidence_text', '')[:1000],
    )
    assistant_part = (
        f'Verdict: {label_ro}\n'
        f'Justificare: {row.get("justification", "")}'
    )
    return (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{user_part}<|end|>\n'
        f'<|assistant|>\n{assistant_part}<|end|>'
    )


def build_inference_prompt(row: pd.Series) -> str:
    user_part = VERDICT_PROMPT_TEMPLATE.format(
        claim=row.get('claim_text', ''),
        evidence=row.get('evidence_text', '')[:1000],
    )
    return (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{user_part}<|end|>\n'
        f'<|assistant|>\n'
    )


# Show one example
sample = train_df.iloc[0]
print(build_training_prompt(sample)[:800])

## 3. Load Model with 4-bit Quantisation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {BASE_MODEL} with 4-bit quantisation …')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1
print('Base model loaded.')

## 4. Attach LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

## 5. Prepare Training Dataset

In [ ]:
from datasets import Dataset

MAX_SEQ_LEN = 1024

def prepare_dataset(df: pd.DataFrame) -> Dataset:
    texts = [build_training_prompt(row) for _, row in df.iterrows()]
    encodings = tokenizer(
        texts,
        max_length=MAX_SEQ_LEN,
        truncation=True,
        padding='max_length',
        return_tensors='pt',
    )
    # Labels = input_ids; pad tokens masked with -100
    labels = encodings['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    ds = Dataset.from_dict({
        'input_ids': encodings['input_ids'].tolist(),
        'attention_mask': encodings['attention_mask'].tolist(),
        'labels': labels.tolist(),
    })
    return ds

train_dataset = prepare_dataset(train_df)
val_dataset   = prepare_dataset(val_df)
print(f'Train dataset size: {len(train_dataset)}')

## 6. LoRA Fine-Tuning with SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=LORA_OUTPUT,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=25,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

trainer.train()
peft_model.save_pretrained(LORA_OUTPUT)
tokenizer.save_pretrained(LORA_OUTPUT)
print(f'LoRA adapters saved to {LORA_OUTPUT}')

## 7. Inference Helper

In [ ]:
def generate_verdict(model, tokenizer_inst, prompt: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer_inst(prompt, return_tensors='pt', truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer_inst.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer_inst.decode(generated, skip_special_tokens=True)

def batch_generate_verdict(model, tokenizer_inst, prompts: list, max_new_tokens: int = 256, batch_size: int = 8) -> list:
    tokenizer_inst.padding_side = 'left'
    results = []
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        inputs = tokenizer_inst(batch_prompts, return_tensors='pt', truncation=True, padding=True, max_length=1024)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer_inst.eos_token_id,
            )
        for j, out in enumerate(outputs):
            prompt_len = inputs['input_ids'][j].shape[0]
            generated = out[prompt_len:]
            results.append(tokenizer_inst.decode(generated, skip_special_tokens=True))
    return results


## 8. Baseline 1 — Zero-Shot Prompting (untuned Llama)

In [ ]:
# Reload base model without LoRA for baselines
from peft import PeftModel

ZERO_SHOT_LIMIT = 200  # evaluate on a subset for speed
zero_shot_df = test_df.sample(min(ZERO_SHOT_LIMIT, len(test_df)), random_state=42).reset_index(drop=True)

zs_preds, zs_true, zs_justifications = [], [], []
prompts = [build_inference_prompt(row) for _, row in zero_shot_df.iterrows()]

outputs = batch_generate_verdict(base_model, tokenizer, prompts, batch_size=8)

for output, (_, row) in zip(outputs, zero_shot_df.iterrows()):
    label, just = parse_verdict(output)
    zs_preds.append(label)
    zs_true.append(row['veracity_label'])
    zs_justifications.append(just)

zs_metrics = classification_metrics(zs_true, zs_preds)
print('=== Zero-Shot Baseline ===')
print_classification_metrics(zs_metrics)


## 9. Baseline 2 — 5-Step Chain-of-Thought (untuned)

In [ ]:
COT_PROMPT_TEMPLATE = """\
<|system|>
Ești un expert verificator de fapte pentru Republica Moldova. Gândește pas cu pas înainte de a da un verdict.
<|end|>
<|user|>
Afirmație: {claim}

Dovezi:
{evidence}

Răspunde urmând cei 5 pași:
1. Ce susține afirmația?
2. Ce informații relevante există în dovezi?
3. Există contradicții sau confirmări directe?
4. Există informații parțiale sau ambigue?
5. Care este verdictul final?

Verdict final: [ADEVĂRAT / FALS / PARȚIAL_ADEVĂRAT]
Justificare: [max 3 propoziții în română]
<|end|>
<|assistant|>
"""

cot_preds, cot_true, cot_justifications = [], [], []
prompts = []
for _, row in zero_shot_df.iterrows():
    prompts.append(COT_PROMPT_TEMPLATE.format(
        claim=row.get('claim_text', ''),
        evidence=row.get('evidence_text', '')[:1000],
    ))

outputs = batch_generate_verdict(base_model, tokenizer, prompts, batch_size=8)

for output, (_, row) in zip(outputs, zero_shot_df.iterrows()):
    label, just = parse_verdict(output)
    cot_preds.append(label)
    cot_true.append(row['veracity_label'])
    cot_justifications.append(just)

cot_metrics = classification_metrics(cot_true, cot_preds)
print('=== 5-Step CoT Baseline ===')
print_classification_metrics(cot_metrics)


## 10. LoRA Fine-Tuned Model Evaluation

In [ ]:
# Load fine-tuned LoRA model
ft_model = PeftModel.from_pretrained(base_model, LORA_OUTPUT)
ft_model.eval()

ft_preds, ft_true, ft_justifications = [], [], []
prompts = [build_inference_prompt(row) for _, row in test_df.iterrows()]

outputs = batch_generate_verdict(ft_model, tokenizer, prompts, batch_size=8)

for output, (_, row) in zip(outputs, test_df.iterrows()):
    label, just = parse_verdict(output)
    ft_preds.append(label)
    ft_true.append(row['veracity_label'])
    ft_justifications.append(just)

ft_metrics = classification_metrics(ft_true, ft_preds)
print('=== LoRA Fine-Tuned Llama ===')
print_classification_metrics(ft_metrics)


## 11. Save Results

In [ ]:
results = {
    'zero_shot': zs_metrics,
    'cot_baseline': cot_metrics,
    'lora_finetuned': ft_metrics,
}

with open(PROCESSED_DIR / 'generative_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# --- Save LoRA predictions (full test set) ---
test_df_copy = test_df.copy().reset_index(drop=True)
test_df_copy['pred_lora'] = ft_preds[:len(test_df_copy)]
test_df_copy['just_lora'] = ft_justifications[:len(test_df_copy)]
test_df_copy.to_csv(PROCESSED_DIR / 'test_with_lora_preds.csv', index=False, encoding='utf-8')

# --- Save zero-shot predictions (subset) ---
# Align claim_id so Notebook 05 can join on it for NLG eval
zs_out = zero_shot_df[['claim_id', 'claim_text', 'veracity_label']].copy()
zs_out['pred_zs']   = zs_preds
zs_out['just_zs']   = zs_justifications
zs_out.to_csv(PROCESSED_DIR / 'test_with_zs_preds.csv', index=False, encoding='utf-8')

# --- Save CoT predictions (same subset) ---
cot_out = zero_shot_df[['claim_id', 'claim_text', 'veracity_label']].copy()
cot_out['pred_cot']  = cot_preds
cot_out['just_cot']  = cot_justifications
cot_out.to_csv(PROCESSED_DIR / 'test_with_cot_preds.csv', index=False, encoding='utf-8')

print('All generative results saved:')
print(f'  generative_results.json')
print(f'  test_with_lora_preds.csv  ({len(test_df_copy)} rows)')
print(f'  test_with_zs_preds.csv    ({len(zs_out)} rows)')
print(f'  test_with_cot_preds.csv   ({len(cot_out)} rows)')